# LAB 2 — Setup do Ambiente Local: VS Code, Python, Ollama e Verificacao

**Curso:** MBA RAG & CAG Aplicados a Direito e Segurança Pública  
**Aula:** 1 — Fundamentos  
**Duracao estimada:** 25 minutos  
**Ambiente:** Jupyter Local no VS Code

---

## Objetivo
Configurar e validar o ambiente Python completo para todas as aulas do curso:
1. Verificar a versao do Python e o ambiente virtual ativo
2. Confirmar que o Ollama esta rodando e com modelos disponiveis
3. Instalar todas as dependencias do curso
4. Validar as instalacoes com um smoke test de embedding
5. Configurar as variaveis de ambiente

---

> **PRE-REQUISITO:** Antes de executar este notebook, siga o `ROTEIRO_INSTALACAO_FERRAMENTAS.md`  
> para instalar VS Code, Python 3.11+, e o Ollama na sua maquina.

---

## Por que Jupyter Local em vez do Google Colab?

Em sistemas RAG para direito e segurança pública, os dados processados frequentemente contêm  
informações sensíveis: nomes de investigados, conteúdo de escutas autorizadas, detalhes de  
procedimentos sigilosos. Enviar esses dados para servidores externos (como o Google Colab)  
pode violar a **LGPD**, o **sigilo funcional** e comprometer a cadeia de custódia das provas.  

O ambiente local garante que **nenhum dado sai da máquina**. O VS Code com a extensão Jupyter  
oferece a mesma experiência de notebooks com controle total sobre os dados.

## CELULA 1 — Verificar Python e Ambiente Virtual

**O que faz:** Verifica a versao do Python, o ambiente virtual ativo e recursos do sistema.  
**Por que:** Garante que estamos usando o interpretador correto (`venv_rag`) antes de qualquer instalacao.

In [ ]:
# CELULA 1: Verificar Python e ambiente
import sys
import platform
import subprocess

print('=' * 60)
print('INFORMACOES DO AMBIENTE')
print('=' * 60)
print(f'Python:    {sys.version}')
print(f'Plataforma: {platform.system()} {platform.release()}')
print(f'Interpretador: {sys.executable}')

# Verificar se esta dentro de um virtualenv
in_venv = hasattr(sys, 'real_prefix') or (
    hasattr(sys, 'base_prefix') and sys.base_prefix != sys.prefix
)
if in_venv:
    print(f'Ambiente virtual: ativo ({sys.prefix})')
else:
    print('AVISO: Nao esta dentro de um ambiente virtual.')
    print('  Recomendado: ative o venv_rag antes de continuar.')
    print('  Linux/macOS: source ~/mba-rag/venv_rag/bin/activate')
    print('  Windows:     .\\venv_rag\\Scripts\\Activate.ps1')

# Verificar versao do Python
if sys.version_info >= (3, 11):
    print(f'Python 3.11+: OK ({sys.version_info.major}.{sys.version_info.minor})')
else:
    print(f'AVISO: Python {sys.version_info.major}.{sys.version_info.minor} detectado.')
    print('  O curso requer Python 3.11+. Instale em: https://python.org')

# Recursos do sistema
try:
    import psutil
    ram = psutil.virtual_memory()
    disk = psutil.disk_usage('/')
    print(f'\nRAM:   {ram.total // (1024**3)} GB total, {ram.available // (1024**3)} GB disponivel')
    print(f'Disco: {disk.free // (1024**3)} GB livre de {disk.total // (1024**3)} GB')
except ImportError:
    print('\n(psutil nao instalado — informacoes de RAM/Disco indisponiveis)')

# GPU
result = subprocess.run(
    ['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'],
    capture_output=True, text=True
)
if result.returncode == 0:
    print(f'\nGPU (NVIDIA): {result.stdout.strip()}')
    print('  Ollama usara a GPU automaticamente.')
else:
    print('\nGPU NVIDIA nao detectada — Ollama rodara em CPU (mais lento, mas funcional).')

## CELULA 2 — Verificar Ollama

**O que faz:** Verifica se o Ollama esta rodando e lista os modelos disponíveis.  
**Por que:** O Ollama e o servidor central de LLMs e embeddings do curso. Sem ele, os labs nao funcionam.

In [ ]:
# CELULA 2: Verificar Ollama
import requests

OLLAMA_BASE_URL = 'http://localhost:11434'

print('VERIFICANDO SERVIDOR OLLAMA')
print('=' * 60)

def verificar_ollama():
    """Verifica se o Ollama esta rodando e lista modelos."""
    try:
        r = requests.get(f'{OLLAMA_BASE_URL}/api/tags', timeout=5)
        if r.status_code == 200:
            modelos = [m['name'] for m in r.json().get('models', [])]
            print('Ollama: ONLINE')
            print(f'URL: {OLLAMA_BASE_URL}')
            if modelos:
                print(f'Modelos instalados ({len(modelos)}):')
                for m in modelos:
                    print(f'  - {m}')
            else:
                print('AVISO: Nenhum modelo instalado ainda.')
                print('  Execute no terminal:')
                print('    ollama pull llama3.2:3b')
                print('    ollama pull nomic-embed-text')
            return modelos
        else:
            print(f'Ollama respondeu HTTP {r.status_code}')
            return []
    except requests.exceptions.ConnectionError:
        print('ERRO: Ollama nao esta rodando!')
        print('  Solucoes:')
        print('  1. Windows: verifique o icone do Ollama na bandeja do sistema')
        print('  2. Qualquer SO: execute no terminal: ollama serve')
        print('  3. Linux: sudo systemctl start ollama')
        return []
    except Exception as e:
        print(f'Erro inesperado: {e}')
        return []

modelos_disponiveis = verificar_ollama()

# Verificar modelos obrigatorios
print()
print('VERIFICACAO DOS MODELOS OBRIGATORIOS:')
MODELOS_OBRIGATORIOS = {
    'llama3.2:3b': 'LLM de geracao de texto (padrão do curso)',
    'nomic-embed-text': 'Modelo de embedding (padrão do curso)',
}
for modelo, descricao in MODELOS_OBRIGATORIOS.items():
    # Verifica por prefixo para capturar variantes como llama3.2:3b-instruct
    encontrado = any(m.startswith(modelo.split(':')[0]) for m in modelos_disponiveis)
    status = 'OK' if encontrado else 'FALTA'
    print(f'  [{status}] {modelo} — {descricao}')
    if not encontrado:
        print(f'         Execute: ollama pull {modelo}')

## CELULA 3 — Instalar Dependencias do Curso

**O que faz:** Instala todas as bibliotecas Python necessárias para os labs.  
**Por que:** Garante que o ambiente tem todas as dependencias antes de prosseguir.

> **Tempo estimado:** 3-8 minutos na primeira execucao. Execute apenas uma vez por ambiente.

In [ ]:
# CELULA 3: Instalar dependencias
# O %pip install funciona no Jupyter local e garante instalacao no kernel correto

print('Instalando dependencias do curso MBA RAG...')
print('Aguarde, pode levar alguns minutos na primeira vez.')
print('=' * 60)

# NLP e Embeddings
print('[1/5] NLP e Embeddings...')
%pip install sentence-transformers==3.0.1 FlagEmbedding==1.2.11 --quiet
print('  OK')

# Busca Vetorial
print('[2/5] Busca Vetorial...')
%pip install faiss-cpu==1.8.0 opensearch-py==2.7.1 --quiet
print('  OK')

# LLM e Agentes
print('[3/5] Frameworks LLM...')
%pip install langchain==0.3.0 langchain-community==0.3.0 langchain-openai==0.2.0 openai>=1.40.0 --quiet
print('  OK')

# Observabilidade
print('[4/5] Observabilidade...')
%pip install langfuse==2.36.1 --quiet
print('  OK')

# Visualizacao e Utilitarios
print('[5/5] Visualizacao e Utilitarios...')
%pip install umap-learn==0.5.6 matplotlib==3.9.2 plotly==5.24.0 seaborn==0.13.2 pandas==2.2.2 numpy==1.26.4 tqdm==4.66.5 python-dotenv==1.0.1 psutil==6.0.0 scikit-learn nltk --quiet
print('  OK')

print()
print('=' * 60)
print('TODAS AS DEPENDENCIAS INSTALADAS!')
print('Execute a proxima celula para validar.')

## CELULA 4 — Validar Instalacoes

**O que faz:** Importa todas as bibliotecas instaladas e verifica suas versoes.  
**Por que:** Conflitos de versao podem causar erros crypticos mais tarde. Melhor verificar agora.

In [ ]:
# CELULA 4: Validar importacoes
print('VALIDANDO INSTALACOES...')
print('=' * 60)

resultados = []

def testar_import(modulo, nome_display=None, attr_versao='__version__'):
    display = nome_display or modulo
    try:
        mod = __import__(modulo)
        versao = getattr(mod, attr_versao, 'N/D')
        resultados.append((display, 'OK', str(versao)))
    except ImportError as e:
        resultados.append((display, 'FALHA', str(e)[:50]))

testar_import('sentence_transformers', 'sentence-transformers')
testar_import('FlagEmbedding', 'FlagEmbedding (BGE-M3)')
testar_import('faiss', 'faiss-cpu')
testar_import('opensearchpy', 'opensearch-py')
testar_import('langchain', 'langchain')
testar_import('openai', 'openai')
testar_import('langfuse', 'langfuse')
testar_import('umap', 'umap-learn')
testar_import('matplotlib', 'matplotlib')
testar_import('pandas', 'pandas')
testar_import('numpy', 'numpy')
testar_import('sklearn', 'scikit-learn')
testar_import('requests', 'requests')

print(f'{"Biblioteca":<30} {"Status":<8} {"Versao"}')
print('-' * 60)
for nome, status, versao in resultados:
    icone = '[OK]  ' if status == 'OK' else '[FALHA]'
    print(f'{nome:<30} {icone:<8} {versao}')

ok = sum(1 for _, s, _ in resultados if s == 'OK')
total = len(resultados)
print(f'\nResultado: {ok}/{total} bibliotecas OK')

if ok == total:
    print('AMBIENTE PYTHON COMPLETAMENTE CONFIGURADO!')
else:
    falhas = [n for n, s, _ in resultados if s == 'FALHA']
    print(f'Problemas em: {falhas}')
    print('Execute a celula 3 novamente ou instale manualmente.')

## CELULA 5 — Configurar Variaveis de Ambiente

**O que faz:** Carrega configuracoes do arquivo `.env` e define variaveis para o curso.  
**Por que:** Centraliza as configuracoes em um arquivo seguro, sem expor chaves no codigo.

In [ ]:
# CELULA 5: Configurar variaveis de ambiente
import os
from pathlib import Path

# Tenta carregar o .env da pasta do projeto
try:
    from dotenv import load_dotenv
    # Procura o .env em possiveis locais
    possiveis_env = [
        Path.home() / 'mba-rag' / '.env',
        Path('.env'),
        Path('..') / '.env',
    ]
    env_carregado = False
    for env_path in possiveis_env:
        if env_path.exists():
            load_dotenv(env_path)
            print(f'Arquivo .env carregado: {env_path.resolve()}')
            env_carregado = True
            break
    if not env_carregado:
        print('Arquivo .env nao encontrado — usando valores padrao.')
        print('Crie ~/mba-rag/.env seguindo o ROTEIRO_INSTALACAO_FERRAMENTAS.md')
except ImportError:
    print('python-dotenv nao instalado — usando apenas variaveis de ambiente existentes.')

# Configuracoes do curso
OLLAMA_BASE_URL  = os.environ.get('OLLAMA_BASE_URL',  'http://localhost:11434')
OLLAMA_LLM_MODEL = os.environ.get('OLLAMA_LLM_MODEL', 'llama3.2:3b')
OLLAMA_EMBED_MODEL = os.environ.get('OLLAMA_EMBED_MODEL', 'nomic-embed-text')
OPENSEARCH_URL   = os.environ.get('OPENSEARCH_URL',   'http://localhost:9200')
LANGFUSE_PUBLIC_KEY = os.environ.get('LANGFUSE_PUBLIC_KEY', '')
LANGFUSE_SECRET_KEY = os.environ.get('LANGFUSE_SECRET_KEY', '')
LANGFUSE_HOST    = os.environ.get('LANGFUSE_HOST', 'https://cloud.langfuse.com')

# Define como variaveis de ambiente para uso pelos frameworks
os.environ['LANGFUSE_PUBLIC_KEY'] = LANGFUSE_PUBLIC_KEY
os.environ['LANGFUSE_SECRET_KEY'] = LANGFUSE_SECRET_KEY
os.environ['LANGFUSE_HOST'] = LANGFUSE_HOST

def status_chave(chave, nome):
    if chave and not chave.endswith('_AQUI') and len(chave) > 10:
        return f'OK ({len(chave)} chars)'
    return 'NAO CONFIGURADA — edite o arquivo .env'

print()
print('CONFIGURACAO DO AMBIENTE')
print('=' * 60)
print(f'Ollama URL:          {OLLAMA_BASE_URL}')
print(f'Ollama LLM:          {OLLAMA_LLM_MODEL}')
print(f'Ollama Embedding:    {OLLAMA_EMBED_MODEL}')
print(f'OpenSearch URL:      {OPENSEARCH_URL}')
print(f'LangFuse Host:       {LANGFUSE_HOST}')
print(f'LangFuse Public Key: {status_chave(LANGFUSE_PUBLIC_KEY, "LANGFUSE_PUBLIC_KEY")}')
print(f'LangFuse Secret Key: {status_chave(LANGFUSE_SECRET_KEY, "LANGFUSE_SECRET_KEY")}')
print()
print('Variaveis configuradas! Continue para o smoke test.')

## CELULA 6 — Smoke Test: Embedding via Ollama

**O que faz:** Gera embeddings de frases juridicas usando o Ollama e calcula similaridade.  
**Por que:** Valida que o pipeline completo (Python → Ollama → embeddings) esta funcionando.

In [ ]:
# CELULA 6: Smoke test de embedding via Ollama
import requests
import numpy as np

print('SMOKE TEST: Embedding via Ollama')
print('=' * 60)

def gerar_embedding(texto: str, modelo: str = OLLAMA_EMBED_MODEL) -> list:
    """Gera embedding de um texto via Ollama API."""
    r = requests.post(
        f'{OLLAMA_BASE_URL}/api/embeddings',
        json={'model': modelo, 'prompt': texto},
        timeout=30
    )
    r.raise_for_status()
    return r.json()['embedding']

def similaridade_coseno(v1: list, v2: list) -> float:
    """Calcula similaridade coseno entre dois vetores."""
    a, b = np.array(v1), np.array(v2)
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))

# Frases de teste com vocabulario juridico
frases = [
    'O reu foi absolvido por insuficiencia de provas.',
    'O acusado foi inocentado por falta de evidencias.',  # Semanticamente similar
    'O tribunal condenou o servidor pelo crime de peculato.',  # Diferente
    'A vitima registrou boletim de ocorrencia na delegacia.'
]

print(f'Modelo de embedding: {OLLAMA_EMBED_MODEL}')
print(f'Gerando embeddings para {len(frases)} frases...')

try:
    embeddings = []
    for i, frase in enumerate(frases):
        emb = gerar_embedding(frase)
        embeddings.append(emb)
        print(f'  [{i+1}/{len(frases)}] OK — {len(emb)} dims')

    print(f'\nDimensao dos embeddings: {len(embeddings[0])}')

    # Calcular similaridades relevantes
    sim_01 = similaridade_coseno(embeddings[0], embeddings[1])
    sim_02 = similaridade_coseno(embeddings[0], embeddings[2])
    sim_03 = similaridade_coseno(embeddings[0], embeddings[3])

    print()
    print('SIMILARIDADE COSENO:')
    print(f'  "Absolvido" vs "Inocentado" (devem ser proximas): {sim_01:.3f}')
    print(f'  "Absolvido" vs "Peculato"   (devem ser distantes):  {sim_02:.3f}')
    print(f'  "Absolvido" vs "Boletim BO" (devem ser distantes):  {sim_03:.3f}')

    if sim_01 > sim_02 and sim_01 > sim_03:
        print()
        print('OK: Embeddings semanticamente corretos!')
        print('  "Absolvido" e "Inocentado" estao mais proximos que os outros pares.')
    else:
        print()
        print('AVISO: Resultado inesperado. Verifique o modelo de embedding.')

except requests.exceptions.ConnectionError:
    print('ERRO: Nao foi possivel conectar ao Ollama.')
    print('  Verifique se o Ollama esta rodando: ollama serve')
except Exception as e:
    print(f'ERRO: {e}')
    print('  Verifique se o modelo esta instalado: ollama pull nomic-embed-text')

## Checklist de Validacao do Lab 2

- [ ] Python 3.11+ verificado e ambiente virtual ativo
- [ ] Ollama rodando em localhost:11434
- [ ] Modelos `llama3.2:3b` e `nomic-embed-text` instalados
- [ ] Todas as dependencias Python instaladas
- [ ] Arquivo `.env` criado com as configuracoes
- [ ] Smoke test de embedding passou (similaridade semantica correta)

**Pontuacao:** 6/6 = Lab 2 completo

## Proximo Passo

Continue para o **LAB3 — Ollama + LLM Local** para explorar geracoes de texto, parametros de geracao e system messages juridicos.